# 04 — Interpretabilidade

Feature importance do Random Forest e SHAP (summary plot, force plot em casos individuais).

In [2]:
# Célula Markdown

# 04 — Interpretabilidade

Este notebook apresenta a etapa de interpretabilidade do modelo de predição de prematuridade.

**Objetivos:**
- Avaliar a importância das variáveis no `RandomForest`;
- Aplicar SHAP para interpretação global e local;
- Gerar `summary plot` e `force plot` em casos individuais;
- Discutir criticamente o uso prático do modelo como apoio à triagem, sem substituir decisão clínica.

SyntaxError: invalid syntax (2620875449.py, line 5)

In [3]:
# Célula Markdown

## 1. Imports e configuração

In [9]:
# Célula de código

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from utils import load_parquet_safe

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("deep")

RANDOM_STATE = 42
N_JOBS = -1

FIGURES_DIR = Path("../results/figures")
METRICS_DIR = Path("../results/metrics")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Amostras para manter o SHAP viável computacionalmente
SHAP_BACKGROUND_SIZE = 300
SHAP_EXPLAIN_SIZE = 1000

np.random.seed(RANDOM_STATE)

print("Configuração concluída.")

Configuração concluída.


C:\Users\aalan\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Célula Markdown

## 2. Carga dos dados pré-processados

Os arquivos utilizados nesta etapa são os artefatos gerados no notebook de preprocessing.

In [10]:
# Célula de código

X_train = load_parquet_safe("../data/X_train.parquet", "02_preprocessing.ipynb")
X_test = load_parquet_safe("../data/X_test.parquet", "02_preprocessing.ipynb")
y_train = load_parquet_safe("../data/y_train.parquet", "02_preprocessing.ipynb").iloc[:, 0]
y_test = load_parquet_safe("../data/y_test.parquet", "02_preprocessing.ipynb").iloc[:, 0]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

display(X_train.head())

FileNotFoundError: Erro: O arquivo '../data/X_train.parquet' está faltando. Por favor, execute o notebook '02_preprocessing.ipynb' antes de continuar.

In [ ]:
# Célula Markdown

## 3. Recriação do Random Forest final

Como o notebook precisa funcionar de forma independente, o `RandomForest` é recriado com os melhores parâmetros obtidos na modelagem.

Na execução anterior, o modelo selecionado foi `RandomForest`, com foco em maior `Recall` e `F2` para a classe positiva `PREMATURO=1`.

In [11]:
# Célula de código

rf_model = RandomForestClassifier(
    n_estimators=250,
    min_samples_leaf=8,
    max_depth=8,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
)

rf_model.fit(X_train, y_train)

y_test_prob = rf_model.predict_proba(X_test)[:, 1]

print("RandomForest treinado.")

NameError: name 'X_train' is not defined

In [ ]:
# Célula Markdown

## 4. Métricas de referência do modelo

Esta seção recalcula métricas básicas para confirmar que o modelo está funcionando corretamente no notebook de interpretabilidade.

São avaliados dois cenários:
- Threshold padrão `0.50`;
- Threshold ajustado `0.41`, definido na etapa de modelagem para priorizar recall.

In [ ]:
# Célula de código

def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "threshold": float(threshold),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "average_precision": average_precision_score(y_true, y_prob),
    }


metrics_default = compute_binary_metrics(y_test, y_test_prob, threshold=0.5)
metrics_tuned = compute_binary_metrics(y_test, y_test_prob, threshold=0.41)

rf_metrics_df = pd.DataFrame([
    {"scenario": "default_0.5", **metrics_default},
    {"scenario": "tuned_threshold_0.41", **metrics_tuned},
])

display(rf_metrics_df)

rf_metrics_df.to_csv(METRICS_DIR / "04_rf_metrics_reference.csv", index=False)
print("Salvo:", METRICS_DIR / "04_rf_metrics_reference.csv")

In [ ]:
# Célula Markdown

## 5. Feature importance do Random Forest

A importância de variáveis do `RandomForest` indica quanto cada feature contribuiu, em média, para a redução de impureza nas árvores.

Essa métrica é útil para uma primeira leitura global, mas possui limitações:
- Pode favorecer variáveis com maior variabilidade ou maior número de pontos de corte;
- Não informa a direção do efeito;
- Não substitui análise causal;
- Deve ser interpretada junto com SHAP e conhecimento clínico.

In [ ]:
# Célula de código

rf_feature_importance = (
    pd.DataFrame({
        "feature": X_train.columns,
        "importance": rf_model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(rf_feature_importance.head(25))

rf_feature_importance.to_csv(METRICS_DIR / "04_rf_feature_importance.csv", index=False)
print("Salvo:", METRICS_DIR / "04_rf_feature_importance.csv")

In [ ]:
# Célula de código

top_n = 20
top_features = rf_feature_importance.head(top_n).sort_values("importance", ascending=True)

plt.figure(figsize=(10, 8))
sns.barplot(
    data=top_features,
    x="importance",
    y="feature",
    color="#4C72B0",
)

plt.title(f"RandomForest — Top {top_n} features por importância")
plt.xlabel("Importância")
plt.ylabel("Feature")
plt.tight_layout()

rf_importance_path = FIGURES_DIR / "04_rf_feature_importance_top20.png"
plt.savefig(rf_importance_path, dpi=150)
plt.show()

print("Salvo:", rf_importance_path)

In [ ]:
# Célula Markdown

### Leitura inicial da feature importance

As variáveis mais relevantes para o `RandomForest` devem ser interpretadas como sinalizadores estatísticos associados ao risco de prematuridade.

Em geral, espera-se que variáveis relacionadas a:
- Tipo de gravidez;
- Adequação do pré-natal;
- Início do pré-natal;
- Idade materna;
- Histórico gestacional;
- Localidade ou variáveis geográficas;

apareçam entre os principais preditores.

Essa leitura sugere que o modelo está capturando tanto fatores obstétricos quanto marcadores de acesso e acompanhamento durante a gestação.

In [ ]:
# Célula Markdown

## 6. Preparação das amostras para SHAP

O SHAP pode ser custoso em bases grandes. Por isso, são usadas amostras controladas:

- `background`: amostra usada como referência pelo explicador;
- `X_shap`: amostra de teste explicada.

A amostragem preserva reprodutibilidade e torna o notebook executável em ambiente local.

In [ ]:
# Célula de código

background_size = min(SHAP_BACKGROUND_SIZE, len(X_train))
explain_size = min(SHAP_EXPLAIN_SIZE, len(X_test))

X_background = X_train.sample(
    n=background_size,
    random_state=RANDOM_STATE,
)

X_shap = X_test.sample(
    n=explain_size,
    random_state=RANDOM_STATE,
)

print("X_background:", X_background.shape)
print("X_shap:", X_shap.shape)

In [ ]:
# Célula Markdown

## 7. SHAP TreeExplainer

Para o `RandomForest`, é utilizado `shap.TreeExplainer`, adequado para modelos baseados em árvores.

Como o problema é binário, o foco será na explicação da classe positiva:

`PREMATURO = 1`

In [ ]:
# Célula de código

explainer = shap.TreeExplainer(
    rf_model,
    data=X_background,
    feature_perturbation="interventional",
    model_output="probability",
)

shap_values_raw = explainer.shap_values(X_shap)

# Compatibilidade entre versões do SHAP:
# - Algumas versões retornam lista [classe_0, classe_1]
# - Outras retornam array 3D
if isinstance(shap_values_raw, list):
    shap_values_class_1 = shap_values_raw[1]
else:
    shap_values_raw = np.asarray(shap_values_raw)
    if shap_values_raw.ndim == 3:
        shap_values_class_1 = shap_values_raw[:, :, 1]
    else:
        shap_values_class_1 = shap_values_raw

print("SHAP values classe positiva:", shap_values_class_1.shape)

In [ ]:
# Célula Markdown

## 8. SHAP summary plot

O `summary plot` mostra, para cada variável:

- A magnitude média de impacto no modelo;
- A dispersão dos impactos por observação;
- A direção aproximada do efeito, representada pela cor do valor da feature.

Valores SHAP positivos aumentam a probabilidade estimada de prematuridade.
Valores SHAP negativos reduzem a probabilidade estimada de prematuridade.

In [ ]:
# Célula de código

plt.figure()
shap.summary_plot(
    shap_values_class_1,
    X_shap,
    plot_type="dot",
    show=False,
    max_display=20,
)

plt.title("SHAP summary plot — Classe positiva PREMATURO=1")
plt.tight_layout()

shap_summary_path = FIGURES_DIR / "04_shap_summary_plot.png"
plt.savefig(shap_summary_path, dpi=150, bbox_inches="tight")
plt.show()

print("Salvo:", shap_summary_path)

In [ ]:
# Célula Markdown

## 9. Importância média via SHAP

Além do gráfico, calcula-se a média do valor absoluto dos SHAP values por feature.

Essa medida representa a contribuição média global de cada variável para a predição da classe positiva.

In [ ]:
# Célula de código

shap_importance = (
    pd.DataFrame({
        "feature": X_shap.columns,
        "mean_abs_shap": np.abs(shap_values_class_1).mean(axis=0),
    })
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

display(shap_importance.head(25))

shap_importance.to_csv(METRICS_DIR / "04_shap_mean_abs_importance.csv", index=False)
print("Salvo:", METRICS_DIR / "04_shap_mean_abs_importance.csv")

In [ ]:
# Célula de código

top_shap = shap_importance.head(20).sort_values("mean_abs_shap", ascending=True)

plt.figure(figsize=(10, 8))
sns.barplot(
    data=top_shap,
    x="mean_abs_shap",
    y="feature",
    color="#55A868",
)

plt.title("SHAP — Top 20 features por impacto médio absoluto")
plt.xlabel("Média do |SHAP value|")
plt.ylabel("Feature")
plt.tight_layout()

shap_bar_path = FIGURES_DIR / "04_shap_mean_abs_importance_top20.png"
plt.savefig(shap_bar_path, dpi=150)
plt.show()

print("Salvo:", shap_bar_path)

In [ ]:
# Célula Markdown

## 10. Comparação: RandomForest importance vs SHAP

A comparação entre as duas abordagens ajuda a verificar se as variáveis mais importantes são consistentes.

- `RandomForest feature_importances_`: mede contribuição média para redução de impureza;
- `SHAP mean_abs_shap`: mede impacto médio na saída probabilística do modelo.

Divergências entre rankings são esperadas, pois as métricas respondem a perguntas diferentes.

In [ ]:
# Célula de código

importance_comparison = (
    rf_feature_importance
    .merge(shap_importance, on="feature", how="inner")
    .assign(
        rf_rank=lambda df: df["importance"].rank(ascending=False, method="dense").astype(int),
        shap_rank=lambda df: df["mean_abs_shap"].rank(ascending=False, method="dense").astype(int),
    )
    .sort_values("shap_rank")
    .reset_index(drop=True)
)

display(importance_comparison.head(25))

importance_comparison.to_csv(METRICS_DIR / "04_importance_comparison_rf_shap.csv", index=False)
print("Salvo:", METRICS_DIR / "04_importance_comparison_rf_shap.csv")

In [ ]:
# Célula Markdown

## 11. SHAP force plot em casos individuais

O `force plot` permite explicar uma predição individual.

Serão selecionados dois casos:
- Um caso com alta probabilidade estimada de prematuridade;
- Um caso com baixa probabilidade estimada de prematuridade.

Esses gráficos são úteis para entender quais features empurraram a predição para cima ou para baixo em cada indivíduo.

In [ ]:
# Célula de código

X_shap_prob = rf_model.predict_proba(X_shap)[:, 1]

high_risk_pos = int(np.argmax(X_shap_prob))
low_risk_pos = int(np.argmin(X_shap_prob))

individual_cases = {
    "high_risk": high_risk_pos,
    "low_risk": low_risk_pos,
}

case_summary_rows = []

for case_name, pos in individual_cases.items():
    case_summary_rows.append({
        "case": case_name,
        "position_in_X_shap": pos,
        "original_index": X_shap.index[pos],
        "predicted_probability_prematuro": X_shap_prob[pos],
    })

case_summary = pd.DataFrame(case_summary_rows)
display(case_summary)

case_summary.to_csv(METRICS_DIR / "04_shap_individual_cases.csv", index=False)
print("Salvo:", METRICS_DIR / "04_shap_individual_cases.csv")

In [ ]:
# Célula de código

shap.initjs()

expected_value = explainer.expected_value

# Compatibilidade entre versões do SHAP
if isinstance(expected_value, (list, np.ndarray)):
    expected_value_class_1 = expected_value[1] if len(expected_value) > 1 else expected_value[0]
else:
    expected_value_class_1 = expected_value

expected_value_class_1

In [ ]:
# Célula Markdown

### 11.1 Caso individual — maior risco estimado

In [ ]:
# Célula de código

high_risk_force_plot = shap.force_plot(
    expected_value_class_1,
    shap_values_class_1[high_risk_pos, :],
    X_shap.iloc[high_risk_pos, :],
    matplotlib=False,
)

high_risk_html_path = FIGURES_DIR / "04_shap_force_plot_high_risk.html"
shap.save_html(str(high_risk_html_path), high_risk_force_plot)

print("Probabilidade estimada:", round(X_shap_prob[high_risk_pos], 4))
print("HTML salvo:", high_risk_html_path)

high_risk_force_plot

In [ ]:
# Célula Markdown

### 11.2 Caso individual — menor risco estimado

In [ ]:
# Célula de código

low_risk_force_plot = shap.force_plot(
    expected_value_class_1,
    shap_values_class_1[low_risk_pos, :],
    X_shap.iloc[low_risk_pos, :],
    matplotlib=False,
)

low_risk_html_path = FIGURES_DIR / "04_shap_force_plot_low_risk.html"
shap.save_html(str(low_risk_html_path), low_risk_force_plot)

print("Probabilidade estimada:", round(X_shap_prob[low_risk_pos], 4))
print("HTML salvo:", low_risk_html_path)

low_risk_force_plot

In [ ]:
# Célula Markdown

## 12. Explicação tabular dos casos individuais

Para complementar o `force plot`, são listadas as features com maior contribuição positiva e negativa em cada caso.

- Contribuição positiva: aumenta a probabilidade estimada de `PREMATURO=1`;
- Contribuição negativa: reduz a probabilidade estimada de `PREMATURO=1`.

In [ ]:
# Célula de código

def explain_individual_case(case_name, pos, top_n=10):
    case_explanation = (
        pd.DataFrame({
            "feature": X_shap.columns,
            "feature_value": X_shap.iloc[pos].values,
            "shap_value": shap_values_class_1[pos],
        })
        .assign(abs_shap=lambda df: df["shap_value"].abs())
        .sort_values("abs_shap", ascending=False)
        .reset_index(drop=True)
    )

    output_path = METRICS_DIR / f"04_shap_case_{case_name}_top_features.csv"
    case_explanation.to_csv(output_path, index=False)

    print(f"Caso: {case_name}")
    print("Probabilidade estimada:", round(X_shap_prob[pos], 4))
    print("Arquivo salvo:", output_path)

    display(case_explanation.head(top_n))

    return case_explanation


high_risk_explanation = explain_individual_case("high_risk", high_risk_pos, top_n=12)
low_risk_explanation = explain_individual_case("low_risk", low_risk_pos, top_n=12)

In [ ]:
# Célula Markdown

## 13. Síntese das features mais preditivas

A análise combinada de `feature_importance` e SHAP indica que as features mais relevantes tendem a se concentrar em alguns grupos:

1. **Características da gestação**
   - Exemplo: tipo de gravidez.
   - Gravidez múltipla costuma estar associada a maior risco de prematuridade.

2. **Pré-natal e adequação do acompanhamento**
   - Exemplo: indicadores derivados do índice de Kotelchuck, mês de início do pré-natal e pré-natal tardio.
   - Esses atributos podem refletir acesso, oportunidade e qualidade do cuidado.

3. **Histórico obstétrico**
   - Exemplo: número de gestações, filhos vivos, perdas fetais ou partos anteriores.
   - Podem capturar padrões de risco reprodutivo.

4. **Características maternas**
   - Exemplo: idade materna e faixas etárias.
   - Extremos de idade podem se associar a maior risco obstétrico.

5. **Localidade**
   - Exemplo: latitude e longitude.
   - Podem atuar como proxies de contexto territorial, acesso à saúde, estrutura assistencial e desigualdades regionais.

É importante destacar que alta importância preditiva não implica causalidade.

In [ ]:
# Célula Markdown

## 14. Discussão crítica: uso prático como apoio à triagem

O modelo deve ser entendido como uma ferramenta de **apoio à triagem**, e não como substituto da decisão clínica.

### Uso prático possível

Em um cenário assistencial, o modelo poderia ajudar a:
- Identificar gestantes ou nascimentos com maior risco estimado de prematuridade;
- Priorizar revisão de casos por equipes de saúde;
- Apoiar ações de vigilância, busca ativa e qualificação do pré-natal;
- Sinalizar padrões populacionais de risco para planejamento em saúde pública.

### Por que não substitui decisão clínica?

A predição é baseada em registros administrativos e variáveis disponíveis na base.
Ela não incorpora toda a complexidade clínica, como:
- Exame físico;
- Ultrassonografia;
- Condições maternas específicas;
- Intercorrências durante a gestação;
- Avaliação médica longitudinal;
- Contexto individual não registrado.

Portanto, a saída do modelo deve ser interpretada como **sinal de alerta probabilístico**, não como diagnóstico.

In [ ]:
# Célula Markdown

## 15. Limitações: viés, representatividade e qualidade dos dados

### 15.1 Viés de registro

A base depende da qualidade do preenchimento dos dados.
Campos ignorados, ausentes ou preenchidos de forma heterogênea podem introduzir ruído e viés.

### 15.2 Representatividade

O modelo aprende padrões presentes na população registrada.
Se determinados grupos sociais, territórios ou perfis assistenciais estiverem sub-representados ou tiverem pior qualidade de registro, a performance pode ser desigual.

### 15.3 Variáveis como proxies sociais

Algumas features podem capturar desigualdades estruturais indiretamente.
Por exemplo, localidade, início do pré-natal ou adequação do acompanhamento podem refletir diferenças de acesso ao sistema de saúde.

### 15.4 Risco de uso indevido

O modelo não deve ser usado para negar atendimento, reduzir cuidado ou automatizar decisões clínicas.
Seu uso mais apropriado é como ferramenta de apoio para ampliar atenção, nunca para restringi-la.

### 15.5 Necessidade de validação externa

Antes de qualquer uso real, seria necessário:
- Validar o modelo em outros períodos;
- Avaliar desempenho por subgrupos;
- Monitorar drift de dados;
- Revisar variáveis sensíveis;
- Submeter a ferramenta à avaliação clínica, ética e institucional.

In [ ]:
# Célula Markdown

## 16. Conclusão do notebook de interpretabilidade

A interpretabilidade mostrou que o `RandomForest` utiliza principalmente variáveis associadas a:

- Tipo de gravidez;
- Acompanhamento pré-natal;
- Histórico gestacional;
- Características maternas;
- Contexto geográfico.

A análise por SHAP complementou a feature importance tradicional ao permitir:
- Avaliação global da magnitude das contribuições;
- Interpretação local de casos individuais;
- Identificação de fatores que aumentam ou reduzem a probabilidade estimada de prematuridade.

Apesar da utilidade para triagem e priorização de cuidado, o modelo possui limitações importantes relacionadas a viés, representatividade e qualidade dos registros.
Assim, sua aplicação deve ser sempre supervisionada por profissionais de saúde e integrada ao contexto clínico-assistencial.

In [ ]:
# Célula Markdown

## 17. Artefatos gerados

Este notebook salva os seguintes arquivos:

### Figuras
- `../results/figures/04_rf_feature_importance_top20.png`
- `../results/figures/04_shap_summary_plot.png`
- `../results/figures/04_shap_mean_abs_importance_top20.png`
- `../results/figures/04_shap_force_plot_high_risk.html`
- `../results/figures/04_shap_force_plot_low_risk.html`

### Métricas e tabelas
- `../results/metrics/04_rf_metrics_reference.csv`
- `../results/metrics/04_rf_feature_importance.csv`
- `../results/metrics/04_shap_mean_abs_importance.csv`
- `../results/metrics/04_importance_comparison_rf_shap.csv`
- `../results/metrics/04_shap_individual_cases.csv`
- `../results/metrics/04_shap_case_high_risk_top_features.csv`
- `../results/metrics/04_shap_case_low_risk_top_features.csv`

In [ ]:
SHAP_BACKGROUND_SIZE = 100
SHAP_EXPLAIN_SIZE = 300

In [ ]:
results/figures/04_shap_force_plot_high_risk.html
results/figures/04_shap_force_plot_low_risk.html